# objetivo 1 - detectar los 10 mejores clientes para el ítem promocionado

In [1]:
import pandas as pd
import numpy as np
import hashlib

from scipy.sparse import coo_matrix
from implicit.als import AlternatingLeastSquares
from implicit.evaluation import ranking_metrics_at_k

c:\Users\nahia\Desktop\metodos cuantitativos\(reto4)\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("Data/matriz.csv")
df.rename(columns={'id_cliente_enc': 'id'}, inplace=True)

df["user_code"] = df["id"].astype("category").cat.codes
df["item_code"] = df["cod_producto"].astype("category").cat.codes

print(f"usuarios únicos:{df['user_code'].nunique()}")
print(f"items únicos:{df['item_code'].nunique()}")
df.head()

usuarios únicos:47580
items únicos:2936


,id,cod_producto,cantidad,user_code,item_code
0,000f749a1a0151a3a8d5b7f1bc774a67e7dfdd904fe321...,1012305,1,6,0
1,002bfef86d56e238c873bf94abde082f0b62e55bad7c5e...,1012305,1,23,0
2,00333e900567e0d781996e85dae5c98b00bf44db9ca67c...,1012305,4,24,0
3,0040f0db50b160deec310e9e3faa83532c871a9efaa5ea...,1012305,2,32,0
4,00443b076854f68eec18d07f8479c822b9c8b112dae8a1...,1012305,2,33,0


In [3]:
obj = pd.read_csv("Data/objetivos_dni.csv")
obj_1 = obj[obj["nombre"] == "objetivo1"]

item_promo = int(obj_1["id"].values[0])

In [10]:
obj

,nombre,explica,id
0,objetivo1,Articulo promocionado,04201510
1,objetivo2,Otros como tu han comprado,94590177D
2,objetivo2,Otros como tu han comprado,26377962K
3,objetivo2,Otros como tu han comprado,88356518V
4,objetivo2,Otros como tu han comprado,14581643B
5,objetivo2,Otros como tu han comprado,36860056A
6,objetivo2,Otros como tu han comprado,31462606D
7,objetivo2,Otros como tu han comprado,62058633X
8,objetivo2,Otros como tu han comprado,44791862Y
9,objetivo2,Otros como tu han comprado,49494869L


In [4]:
codigo_real_items = df["cod_producto"].astype("category").cat.categories
item2idx = {v: k for k, v in enumerate(codigo_real_items)}
idx2item = dict(enumerate(codigo_real_items))

codigo_real_users = df["id"].astype("category").cat.categories
idx2user = dict(enumerate(codigo_real_users))

item_promo_idx = item2idx[item_promo]

Matriz sparse

In [5]:
def build_matrix(data, n_users, n_items):
    matrix = coo_matrix(
        (np.ones(len(data)), (data['user_code'].values, data['item_code'].values)),
        shape=(n_users, n_items)
    )
    return matrix.tocsr()

n_users = int(df['user_code'].max()) + 1
n_items = int(df['item_code'].max()) + 1


In [6]:

train = df.iloc[:int(0.8 * len(df))]
test  = df.iloc[int(0.8 * len(df)):]

train_matrix = build_matrix(train, n_users, n_items)
test_matrix  = build_matrix(test,  n_users, n_items)

In [ ]:
train_matrix_T = train_matrix.T.tocsr()
test_matrix_T  = test_matrix.T.tocsr()

model_T = AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=40,
    alpha=20,
    random_state=42
)
model_T.fit(train_matrix_T)

c:\Users\nahia\Desktop\metodos cuantitativos\(reto4)\lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 40/40 [01:15<00:00,  1.88s/it]


In [8]:
ids, scores = model_T.recommend(item_promo_idx,
                                train_matrix_T[item_promo_idx],
                                N=10,filter_already_liked_items=True)

df_resultado = pd.DataFrame({
    'ranking':     range(1, 11),
    'user_idx':    ids,
    'id_cliente':  [idx2user[i] for i in ids],
    'score':       scores
})

df_resultado

,ranking,user_idx,id_cliente,score
0,1,26383,8e0da30d9c0813559b5eac8929b68f9bbe8acd506d248b...,1.386598
1,2,40628,da8be58882dd0dd3fbd34ca26ae26ac96c5a1bd7e33b4e...,1.354420
2,3,23623,7f70f2dc3d25ff0e14feea6a7e83dacfcaa79dd1a4b8f1...,1.339881
3,4,6479,233eb825da8b59a782a7f768ed547f350ffcd2ce7e5050...,1.333965
4,5,23868,80a35e5a50fb774ddb4515cbff46a9b2ac81e57aaf7e12...,1.332537
5,6,19289,6817e847de6e8bf5d33ec27e0b01ed1fa79df64a1e9cca...,1.330284
6,7,37515,c978699507f101e1e3f112cb179ef76da295183fc6812b...,1.310440
7,8,753,042d868e0e1772a2b208ff280f3d5a967af02835a33bc0...,1.302064
8,9,1666,0919ba26b1ba3b07e8d3a6ccd4e5ba33194b034e168398...,1.300540
9,10,45580,f5251c955149ef3958bb6470654c9bdc1642e98b4fbdc5...,1.286695


In [9]:
metricas = ranking_metrics_at_k(model_T, train_matrix_T, test_matrix_T, K=10)
print("Métricas de evaluación (K=10):")
for k, v in metricas.items():
    print(f"  {k}: {v:.4f}")

100%|██████████| 1054/1054 [00:00<00:00, 7104.88it/s]

Métricas de evaluación (K=10):
  precision: 0.0090
  map: 0.0016
  ndcg: 0.0064
  auc: 0.5000
